In [1]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

MODEL_PATH = "vit_final_model"
FINETUNED_MODEL_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/vit_final_model"  
processor = AutoImageProcessor.from_pretrained(FINETUNED_MODEL_PATH)
model = AutoModelForImageClassification.from_pretrained(FINETUNED_MODEL_PATH)

print(model.config.id2label)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

{0: 'NonViolence', 1: 'Violence'}


## ViT Model Alone


In [10]:
import cv2
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
from IPython.display import Video, display
import gc

# ---------------- CONFIG ----------------
# FINETUNED_MODEL_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/vit_final_model"
VIDEO_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/EDA/Weapon Used Video/Mother of heavy weapons🥵🔥#mmg#nsg#army#indianarmy#proudindian#nsgcommando#blackcats#specialforces.mp4"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL = 30   # ~1 frame per second (for 30 FPS video)
# ---------------------------------------

print("🖥️ Using device:", DEVICE)

# Load model & processor
processor = AutoImageProcessor.from_pretrained(FINETUNED_MODEL_PATH)
model = AutoModelForImageClassification.from_pretrained(FINETUNED_MODEL_PATH)
model.to(DEVICE)
model.eval()

# Label mapping
id2label = model.config.id2label
print("🔖 Labels:", id2label)

# Storage for probabilities
prob_storage = {label: [] for label in id2label.values()}

# Show selected test video 🎥
print("\n🎬 TEST VIDEO PREVIEW")
display(Video(VIDEO_PATH, embed=True))

# Read video
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:
            # BGR → RGB → PIL
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            # Preprocess
            inputs = processor(image, return_tensors="pt")
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            # Inference
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

            # Store probabilities
            for idx, prob in enumerate(probs):
                label = id2label[idx]
                prob_storage[label].append(prob)

        frame_count += 1

cap.release()

# Clear GPU cache safely
gc.collect()
torch.cuda.empty_cache()

# ---------------- RESULTS ----------------
avg_probs = {
    label: (np.mean(values) * 100 if values else 0.0)
    for label, values in prob_storage.items()
}

final_label = max(avg_probs, key=avg_probs.get)

print("\n🎥 VIDEO ANALYSIS RESULT")
for label, prob in avg_probs.items():
    print(f"{label:15}: {prob:.2f}%")

print(f"\n✅ FINAL PREDICTION: {final_label.upper()}")


🖥️ Using device: cuda


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

🔖 Labels: {0: 'NonViolence', 1: 'Violence'}

🎬 TEST VIDEO PREVIEW



🎥 VIDEO ANALYSIS RESULT
NonViolence    : 70.79%
Violence       : 29.21%

✅ FINAL PREDICTION: NONVIOLENCE


## DETR Alone

In [6]:
import cv2
import torch
import numpy as np
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import Image
from IPython.display import Video, display
import gc

# ---------------- CONFIG ----------------
VIDEO_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/EDA/NonViolence/NV_1000.mp4"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL =10
DETR_THRESHOLD = 0.4
MIN_WEAPON_FRAMES = 3   # Minimum frames weapon must appear
# ---------------------------------------

print("🖥️ Using device:", DEVICE)

# ---------------- LOAD DETR ----------------
detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
detr_model.to(DEVICE)
detr_model.eval()

print("🔍 DETR Loaded Successfully")

# COCO weapon-like classes
violent_objects = ["knife", "baseball bat", "sports ball"]

weapon_frame_count = 0
all_detected_objects = []

# Preview Video
print("\n🎬 TEST VIDEO PREVIEW")
display(Video(VIDEO_PATH, embed=True))

# ---------------- VIDEO PROCESSING ----------------
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            detr_inputs = detr_processor(images=image, return_tensors="pt")
            detr_inputs = {k: v.to(DEVICE) for k, v in detr_inputs.items()}

            outputs = detr_model(**detr_inputs)

            target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)

            results = detr_processor.post_process_object_detection(
                outputs,
                target_sizes=target_sizes,
                threshold=DETR_THRESHOLD
            )[0]

            frame_has_weapon = False

            for score, label, box in zip(
                results["scores"],
                results["labels"],
                results["boxes"]
            ):
                class_name = detr_model.config.id2label[label.item()]
                all_detected_objects.append(class_name)

                if class_name in violent_objects:
                    frame_has_weapon = True

            if frame_has_weapon:
                weapon_frame_count += 1

        frame_count += 1

cap.release()

gc.collect()
torch.cuda.empty_cache()

# ---------------- FINAL DECISION ----------------

unique_objects = list(set(all_detected_objects))

if weapon_frame_count >= MIN_WEAPON_FRAMES:
    final_decision = "🚨 VIOLENCE DETECTED (Weapon Present)"
else:
    final_decision = "✅ NON-VIOLENCE"

print("\n🔎 Detected Objects:", unique_objects)
print("🔢 Weapon Appeared In Frames:", weapon_frame_count)
print("\n🏁 FINAL DECISION:", final_decision)

🖥️ Using device: cuda


Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 DETR Loaded Successfully

🎬 TEST VIDEO PREVIEW



🔎 Detected Objects: ['dining table', 'cell phone', 'person', 'traffic light', 'potted plant', 'chair']
🔢 Weapon Appeared In Frames: 0

🏁 FINAL DECISION: ✅ NON-VIOLENCE


## ViT+YOLO


In [13]:
import cv2
import torch
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from ultralytics import YOLO
from PIL import Image
from IPython.display import Video, display
import gc

# ---------------- CONFIG ----------------
ViT_MODEL_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/vit_final_model"

VIDEO_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Data/Video/EDA/Weapon Used Video/Mother of heavy weapons🥵🔥#mmg#nsg#army#indianarmy#proudindian#nsgcommando#blackcats#specialforces.mp4"

YOLO_MODEL_PATH = "/media/rajendraprasath-m/New Volume/Projects/Final Year Project/Model/yolo_weapon_model.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL = 30
# ---------------------------------------

print("🖥️ Using device:", DEVICE)

# ---------------- LOAD MODELS ----------------

# ViT model
processor = AutoImageProcessor.from_pretrained(ViT_MODEL_PATH)
vit_model = AutoModelForImageClassification.from_pretrained(ViT_MODEL_PATH).to(DEVICE)
vit_model.eval()

id2label = vit_model.config.id2label
print("🔖 ViT Labels:", id2label)

# YOLO model
yolo_model = YOLO(YOLO_MODEL_PATH)
print("🔍 YOLO Model Loaded")

# violent objects list
violent_objects = {"knife","scissors","baseball bat"}

# store detected objects
detected_objects = set()

# store ViT probabilities
prob_storage = {label: [] for label in id2label.values()}

# ---------------- SHOW VIDEO ----------------

print("\n🎬 TEST VIDEO PREVIEW")
display(Video(VIDEO_PATH, embed=True))

# ---------------- PROCESS VIDEO ----------------

cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

with torch.no_grad():

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        # analyze every N frames
        if frame_count % FRAME_INTERVAL == 0:

            # ---------------- YOLO OBJECT DETECTION ----------------

            yolo_results = yolo_model(frame, conf=0.4, verbose=False)

            for r in yolo_results:
                for box in r.boxes:

                    class_id = int(box.cls[0])
                    class_name = yolo_model.names[class_id]

                    detected_objects.add(class_name)

            # ---------------- ViT ACTION CLASSIFICATION ----------------

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            inputs = processor(image, return_tensors="pt")
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            outputs = vit_model(**inputs)

            probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

            for idx, prob in enumerate(probs):

                label = id2label[idx]
                prob_storage[label].append(prob)

        frame_count += 1

cap.release()

gc.collect()
torch.cuda.empty_cache()

# ---------------- ViT RESULTS ----------------

avg_probs = {
    label: (np.mean(values) * 100 if values else 0.0)
    for label, values in prob_storage.items()
}

final_label = max(avg_probs, key=avg_probs.get)

# ---------------- OBJECT RESULTS ----------------

weapon_found = detected_objects.intersection(violent_objects)

# ---------------- FINAL OUTPUT ----------------

print("\n🎥 VIDEO ANALYSIS RESULT")

print("\n📊 ViT Violence Prediction")
for label, prob in avg_probs.items():
    print(f"{label:15}: {prob:.2f}%")

print(f"\n✅ FINAL ACTION PREDICTION: {final_label.upper()}")

print("\n🔎 Objects Detected in Video:")
print(detected_objects)

if weapon_found:
    print("\n⚠ Violent Objects Detected:")
    print(weapon_found)
else:
    print("\n✅ No Violent Objects Found")

🖥️ Using device: cuda


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

🔖 ViT Labels: {0: 'NonViolence', 1: 'Violence'}
🔍 YOLO Model Loaded

🎬 TEST VIDEO PREVIEW



🎥 VIDEO ANALYSIS RESULT

📊 ViT Violence Prediction
NonViolence    : 70.79%
Violence       : 29.21%

✅ FINAL ACTION PREDICTION: NONVIOLENCE

🔎 Objects Detected in Video:
{'suitcase', 'person', 'bicycle', 'keyboard', 'motorcycle'}

✅ No Violent Objects Found
